# Hi Clip Worker — non-black cinematic clips
Simple proven ffmpeg filters only (noise + eq). No broken geq graphs.


In [ ]:
import json, time, pathlib, subprocess, shutil
WORK = pathlib.Path('/kaggle/working')
CLIPS = WORK / 'clips'
CLIPS.mkdir(parents=True, exist_ok=True)

def find_job():
    for p in [WORK / 'job.json', pathlib.Path('job.json')]:
        if p.exists():
            return json.loads(p.read_text())
    root = pathlib.Path('/kaggle/input')
    if root.exists():
        for p in root.rglob('job.json'):
            try:
                return json.loads(p.read_text())
            except Exception:
                pass
    return {'job_id': 'dry-' + str(int(time.time())), 'channel': 'storm', 'clip_seconds': 5, 'num_clips': 3,
            'prompts': ['storm', 'ocean', 'tempest']}

job = find_job()
(WORK / 'job_echo.json').write_text(json.dumps(job, indent=2))
print('JOB', job)


In [ ]:
sec = float(job.get('clip_seconds') or 5)
n = int(job.get('num_clips') or 3)
prompts = job.get('prompts') or ['storm']
presets = [
    ('0x061018', 28, 1.28, 0.35, -0.06),
    ('0x0a1a30', 24, 1.20, 0.50, -0.04),
    ('0x120c22', 22, 1.22, 0.55, -0.05),
    ('0x031525', 16, 1.15, 0.70, -0.02),
]
manifest = []
for i in range(n):
    out = CLIPS / f'clip_{i:02d}.mp4'
    color, noise, contrast, sat, bright = presets[i % len(presets)]
    prompt = prompts[i % len(prompts)]
    vf = f'noise=alls={noise}:allf=t,eq=contrast={contrast}:brightness={bright}:saturation={sat},fps=30'
    cmd = [
        'ffmpeg', '-y', '-f', 'lavfi', '-i', f'color=c={color}:s=1080x1920:d={sec}',
        '-vf', vf, '-c:v', 'libx264', '-preset', 'veryfast', '-crf', '20',
        '-pix_fmt', 'yuv420p', '-t', str(sec), str(out)
    ]
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0 or not out.exists() or out.stat().st_size < 40000:
        print('retry simple', (r.stderr or '')[-300:])
        subprocess.run([
            'ffmpeg', '-y', '-f', 'lavfi', '-i', f'color=c={color}:s=1080x1920:d={sec}',
            '-vf', 'noise=alls=25:allf=t,eq=contrast=1.2,fps=24',
            '-c:v', 'libx264', '-pix_fmt', 'yuv420p', '-t', str(sec), str(out)
        ], capture_output=True)
    root = WORK / f'clip_{i:02d}.mp4'
    if out.exists():
        shutil.copy2(out, root)
    ok = out.exists() and out.stat().st_size > 40000
    manifest.append({'index': i, 'path': out.name, 'prompt': prompt, 'ok': ok,
                     'bytes': out.stat().st_size if out.exists() else 0})
    print('clip', i, ok, out.stat().st_size if out.exists() else 0)

# stitch if 2+
good = [CLIPS / m['path'] for m in manifest if m['ok']]
if len(good) >= 2:
    lst = WORK / 'concat.txt'
    lst.write_text(''.join(f"file '{p}'\n" for p in good))
    master = WORK / 'storm_master.mp4'
    subprocess.run(['ffmpeg', '-y', '-f', 'concat', '-safe', '0', '-i', str(lst),
                    '-c', 'copy', str(master)], capture_output=True)
    print('master', master.exists(), master.stat().st_size if master.exists() else 0)

obj = {'job_id': job.get('job_id'), 'channel': job.get('channel'), 'clips': manifest, 'status': 'complete'}
(WORK / 'manifest.json').write_text(json.dumps(obj, indent=2))
print('DONE', obj)
